## Coding Challenge 5: Web scrapping of an API: COMTRADE

In this notebook, I connect to the Comtrade API to download information of Chilean exports, disagreggated by product.

Based on: https://github.com/uncomtrade/comtradeapicall/blob/main/tests/example%20calling%20functions%20-%20notebook.ipynb

In [ ]:
pip install -q comtradeapicall # Installing COMTRADE package to download their API data.

Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#'

[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [52]:
pip install -q squarify

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly
import plotly.io as pio
import squarify
import requests
import comtradeapicall
import json

In [3]:
subscription_key = '<YOUR KEY>' # comtrade api subscription key (from comtradedeveloper.un.org)
directory = '<OUTPUT DIR>'  # output directory for downloaded files 
proxy_url = '<PROXY URL>'  # optional if you need proxy url

In [4]:
#set some variables
from datetime import date
from datetime import timedelta
today = date.today()
yesterday = today - timedelta(days=1)
lastweek = today - timedelta(days=7)

# Chilean exports during 2023.

Type of Product – Commodities (C) or Services (S)

Frequency – Annual (A) or Monthly (M) 

Classification Code – HS, SITC, BEC for Goods, EBOPS for Services

Period – YYYY for Annual or YYYYMM for Monthly

Reporter – All or Individual Reporters

Publication Date – Publication date range

aggregateBy - The level of disaggregation that you want in the products

### API call

In [5]:
#Here, I call the API for Chilean Exports

exp_chile = comtradeapicall.previewFinalData(typeCode='C', freqCode='A', clCode='HS', period='2023',
                                        reporterCode='152', cmdCode=None, flowCode='X', partnerCode='0', #152 is Chile's country code
                                        partner2Code=None,
                                        customsCode=None, motCode=None, maxRecords=None, format_output='JSON',
                                        aggregateBy=None, breakdownMode='classic', countOnly=None, includeDesc=True)

In [234]:
exp_chile.head()

,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,A,20230101,2023,52,2023,152,CHL,Chile,X,...,None,False,0.0,False,None,7.188690e+05,7.188690e+05,0,False,True
1,C,A,20230101,2023,52,2023,152,CHL,Chile,X,...,None,False,0.0,False,None,3.121716e+05,3.121716e+05,0,False,True
2,C,A,20230101,2023,52,2023,152,CHL,Chile,X,...,None,False,0.0,False,None,2.267025e+06,2.267025e+06,0,False,True
3,C,A,20230101,2023,52,2023,152,CHL,Chile,X,...,None,False,0.0,False,None,1.311351e+07,1.311351e+07,0,False,True
4,C,A,20230101,2023,52,2023,152,CHL,Chile,X,...,None,False,0.0,False,None,3.801520e+05,3.801520e+05,0,False,True


### Cleaning and filtering process

In [92]:
exp_chile = exp_chile[['period', 'partnerDesc', 'cmdCode','cmdDesc', 'aggrLevel', 'fobvalue', 'primaryValue']]

# Extract first 2 digits of product code to aggregate them after
exp_chile['HS2'] = exp_chile['cmdCode'].astype(str).str[:2]

In [93]:
exp_chile.head()

,period,partnerDesc,cmdCode,cmdDesc,aggrLevel,fobvalue,primaryValue,HS2
0,2023,World,3403,Lubricating preparations and those used in oil...,4,7.188690e+05,7.188690e+05,34
1,2023,World,2933,Heterocyclic compounds with nitrogen hetero-at...,4,3.121716e+05,3.121716e+05,29
2,2023,World,7616,Aluminium; articles n.e.c. in chapter 76,4,2.267025e+06,2.267025e+06,76
3,2023,World,060220,"Plants, live; edible fruit or nut trees, shrub...",6,1.311351e+07,1.311351e+07,06
4,2023,World,847910,Machinery and mechanical appliances; for publi...,6,3.801520e+05,3.801520e+05,84


In [94]:
exp_chile_hs2 = exp_chile.groupby('HS2').agg({
    'period': 'first',  # Replace year 2023
    'partnerDesc': 'first',  # Replace partnerDesc with World
    'fobvalue': 'sum',
    'primaryValue': 'sum',
})

exp_chile_hs2.head()

,period,partnerDesc,fobvalue,primaryValue
HS2,,,,
01,2023,World,1.961690e+07,1.961690e+07
02,2023,World,2.249886e+09,2.249886e+09
03,2023,World,7.305868e+09,7.305868e+09
04,2023,World,1.960884e+08,1.960884e+08
05,2023,World,3.271020e+07,3.271020e+07


In [95]:
# In share of total:
exp_chile_hs2['percent_fobvalue'] = exp_chile_hs2['fobvalue'] / exp_chile_hs2['fobvalue'].sum()
exp_chile_hs2 = exp_chile_hs2.reset_index()

In [ ]:
# Merge with HS2 codes
hs = pd.read_excel('HSCodeandDescription.xlsx')

hs.head()

hs.rename(columns={
    'Code': 'HS2',
    'Description': 'description'
}, inplace=True)

hs = hs[['HS2', 'description']]

exp_chile_hs2_merged = pd.merge(exp_chile_hs2, hs, on='HS2', how='left')
exp_chile_hs2_merged.head()

# Split the 'Contains' column by commas and expand it into separate columns
exp_chile_hs2_merged['desc']= exp_chile_hs2_merged['description'].str.split(',', expand=False).str[0]
exp_chile_hs2_merged['desc']= exp_chile_hs2_merged['description'].str.split(';', expand=False).str[0]

,HS2,period,partnerDesc,fobvalue,primaryValue,percent_fobvalue,description,desc
0,01,2023,World,1.961690e+07,1.961690e+07,0.000156,Animals; live,Animals
1,02,2023,World,2.249886e+09,2.249886e+09,0.017925,Meat and edible meat offal,Meat and edible meat offal
2,03,2023,World,7.305868e+09,7.305868e+09,0.058205,"Fish and crustaceans, molluscs and other aquat...","Fish and crustaceans, molluscs and other aquat..."
3,04,2023,World,1.960884e+08,1.960884e+08,0.001562,Dairy produce; birds' eggs; natural honey; edi...,Dairy produce
4,05,2023,World,3.271020e+07,3.271020e+07,0.000261,Animal originated products; not elsewhere spec...,Animal originated products


### Plotting

In [146]:
#Plotting a tree map

# Define a color palette
color_palette = px.colors.qualitative.T10

# Create a color map based on groups
color_map = {}
for i in range(1, 99):
    if 1 <= i <= 9:
        color_map[f'Item{i}'] = color_palette[0]
    elif 10 <= i <= 19:
        color_map[f'Item{i}'] = color_palette[1]
    elif 20 <= i <= 29:
        color_map[f'Item{i}'] = color_palette[2]
    elif 30 <= i <= 39:
        color_map[f'Item{i}'] = color_palette[3]
    elif 40 <= i <= 49:
        color_map[f'Item{i}'] = color_palette[4]
    elif 50 <= i <= 59:
        color_map[f'Item{i}'] = color_palette[5]
    elif 60 <= i <= 69:
        color_map[f'Item{i}'] = color_palette[6]
    elif 70 <= i <= 79:
        color_map[f'Item{i}'] = color_palette[7]
    elif 80 <= i <= 89:
        color_map[f'Item{i}'] = color_palette[8]
    elif 90 <= i <= 99:
        color_map[f'Item{i}'] = color_palette[9]
    elif id == '2023':
        color_map[f'Item{i}'] = 'white'


# Plotting the tree map
fig = px.treemap(exp_chile_hs2_merged, path=['period', 'desc'],
                 values='percent_fobvalue',
                 color='HS2',
                 color_discrete_map=color_map,
                 hover_name='description',
                )

# Add title and subtitle
fig.update_layout(
    title={
        'text': "Chilean Exports for 2023",
        'y':0.95,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': {'size': 24} 
    },
    annotations=[
        dict(
            text="% of total FOB value exported",
            x=0.5,
            y=1.15,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(size=20)
        ),
          dict(
            text="Source: Comtrade API",
            x=0,
            y=-0.4,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(size=100)
        )
    ],
    margin=dict(t=100, l=25, r=25, b=25),
    showlegend=True
)

# Erase the border of the treemap graph
fig.update_traces(marker=dict(line=dict(width=0)))
fig.update_layout(xaxis_showgrid=False, yaxis_showgrid=False)

# Show percentages in each quadrant
fig.update_traces(textinfo='label+percent entry', textfont_size=20)


fig.show()

### Saving the plot

In [ ]:
pio.write_json(fig, '../figures/figure5.json', pretty=True) # JSON
pio.write_html(fig, '../figures/figure5.html') #HTML 